# Stage 03 — Python Fundamentals

Demonstrates the toolkit the rest of the project is built on — **NumPy** vectorised
ops, **pandas** DataFrames, and the reusable helpers in [`src/utils.py`](../src/utils.py)
— on **dummy data**. The real dataset arrives in Stage 04.

Run top to bottom: *Kernel → Restart & Run All*.

In [1]:
# run me first: work from the project root so `from src...` imports resolve
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working from:", ROOT.name)

working from: project


## 1. NumPy — elementwise and vectorised

Pull odds in this project span 1:3 to 1:2,000,000, so most work happens on a `log10`
scale.

In [2]:
import numpy as np

odds = np.array([3, 12, 99, 499, 30_000, 2_000_000])   # 1 : N packs
log_odds = np.log10(odds)
hits_per_box = 24 / odds                                # a 24-pack box

print("odds        ", odds)
print("log10(odds) ", np.round(log_odds, 2))
print("E[hits/box] ", np.round(hits_per_box, 6))

odds         [      3      12      99     499   30000 2000000]
log10(odds)  [0.48 1.08 2.   2.7  4.48 6.3 ]
E[hits/box]  [8.00000e+00 2.00000e+00 2.42424e-01 4.80960e-02 8.00000e-04 1.20000e-05]


In [3]:
# loop vs vectorised: same answer, very different cost
n = 1_000_000
arr = np.arange(n)

print("match:", np.array_equal([x * 2 for x in arr], arr * 2))
%timeit [x * 2 for x in arr]
%timeit arr * 2

match: True
44.1 ms ± 1.16 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
267 μs ± 3.05 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 2. pandas — a toy tier table

A stand-in for the real `card_tiers` table, with the messy column names and mixed-type
numeric columns typical of hand-transcribed CSVs.

In [4]:
import pandas as pd

raw = pd.DataFrame({
    "Tier Name":     ["Base", "Refractor", "Blue /150", "Gold /50", "Auto", "SuperFractor 1/1"],
    "Tier Group":    ["base", "parallel", "parallel", "parallel", "auto", "parallel"],
    "Odds (1:N)":    ["3", "12", "99", "499", "1500", "30000"],
    "Print Run":     ["", "", "150", "50", "", "1"],
    "Est Value ($)": ["0.10", "2.5", "18", "55", "120", "n/a"],
})
raw

,Tier Name,Tier Group,Odds (1:N),Print Run,Est Value ($)
0,Base,base,3,,0.10
1,Refractor,parallel,12,,2.5
2,Blue /150,parallel,99,150,18
3,Gold /50,parallel,499,50,55
4,Auto,auto,1500,,120
5,SuperFractor 1/1,parallel,30000,1,n/a


In [5]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Tier Name      6 non-null      str  
 1   Tier Group     6 non-null      str  
 2   Odds (1:N)     6 non-null      str  
 3   Print Run      6 non-null      str  
 4   Est Value ($)  6 non-null      str  
dtypes: str(5)
memory usage: 502.0 bytes


In [6]:
raw.head()

,Tier Name,Tier Group,Odds (1:N),Print Run,Est Value ($)
0,Base,base,3,,0.10
1,Refractor,parallel,12,,2.5
2,Blue /150,parallel,99,150,18
3,Gold /50,parallel,499,50,55
4,Auto,auto,1500,,120


## 3. Clean and summarise with `src/utils`

`clean_column_names` -> snake_case; `coerce_numeric` turns the string / `'n/a'` columns
into numbers (bad values become `NaN`); `summary_stats` returns a table.

In [7]:
from src.utils import clean_column_names, coerce_numeric, summary_stats, group_summary

df = clean_column_names(raw)
df = coerce_numeric(df, ["odds_1_n", "print_run", "est_value"])
df["log_odds"] = np.log10(df["odds_1_n"])
df

,tier_name,tier_group,odds_1_n,print_run,est_value,log_odds
0,Base,base,3,NaN,0.1,0.477121
1,Refractor,parallel,12,NaN,2.5,1.079181
2,Blue /150,parallel,99,150.0,18.0,1.995635
3,Gold /50,parallel,499,50.0,55.0,2.698101
4,Auto,auto,1500,NaN,120.0,3.176091
5,SuperFractor 1/1,parallel,30000,1.0,NaN,4.477121


In [8]:
summary_stats(df)

,count,missing,mean,std,min,median,max
odds_1_n,6,0,5352.166667,12088.298696,3.000000,299.000000,30000.000000
print_run,3,3,67.000000,75.940766,1.000000,50.000000,150.000000
est_value,5,1,39.120000,50.257805,0.100000,18.000000,120.000000
log_odds,6,0,2.317208,1.453962,0.477121,2.346868,4.477121


## 4. Groupby aggregation

Mean numeric values per `tier_group` — the same move used later for EDA and feature checks.

In [9]:
group_summary(df, by="tier_group")

,tier_group,odds_1_n,print_run,est_value,log_odds
0,auto,1500.0,NaN,120.000000,3.176091
1,base,3.0,NaN,0.100000,0.477121
2,parallel,7652.5,67.0,25.166667,2.562510


## 5. How this carries forward

- `clean_column_names` / `coerce_numeric` -> Stage 04 ingestion and Stage 06 cleaning.
- `summary_stats` -> Stage 08 EDA profiling.
- `group_summary` -> per-`tier_group` breakdowns in Stage 08-09.
- `log10(odds)` -> the `log_odds_pack` feature in Stage 09.

All of it lives in `src/utils.py`, imported rather than re-pasted.